# 02 — Train (GPU T4 ×2 notebook)

**Settings:** Accelerator = **GPU T4 x2**, Internet = On, Input = dataset `pygpt-python-tokens` (+ `pygpt-code` if not cloning from GitHub).

Set `MODE` below and run all cells. Order on Day 1:

| MODE | What it does | Time |
|---|---|---|
| `sanity` | unit checks + 60-step smoke run on synthetic data (both GPUs) | ~3 min |
| `bench` | main config on real data for 60 steps → prints **tok/s** and **peak memory**; tests kill + resume | ~5 min |
| `main` | the 10-hour run — use **Save & Run All** so it survives closing the browser | 10 h |
| `resume` | continue a run from a previous version's `ckpt_latest.pt` (Appendix D of PROJECT.md) | — |

After `bench`, copy the printed tok/s into `MAX_ITERS` below before running `main`.

In [ ]:
# --- Get the code into /kaggle/working/pygpt ---------------------------------
# Primary: clone the public repo (always current). Fallback: the "pygpt-code"
# Kaggle Dataset (works with Internet off / if GitHub is unreachable).
REPO_URL = "https://github.com/Rehan123-bash/pygpt.git"
import os, shutil, subprocess
os.chdir("/kaggle/working")
if os.path.exists("pygpt"):
    shutil.rmtree("pygpt")
if REPO_URL:
    try:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "pygpt"], check=True)
    except Exception as e:
        print("clone failed, falling back to the pygpt-code dataset:", e)
if not os.path.exists("pygpt/model.py"):
    src = "/kaggle/input/pygpt-code"
    for c in (os.path.join(src, "pygpt"), src):
        if os.path.exists(os.path.join(c, "model.py")):
            shutil.copytree(c, "pygpt"); break
    else:
        raise SystemExit("no code: clone failed and no 'pygpt-code' dataset input")
print(sorted(os.listdir("pygpt")))

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("torch", torch.__version__, "| GPUs:", torch.cuda.device_count())

In [ ]:
# ================= SET THESE =================
MODE = "sanity"                 # "sanity" | "bench" | "main" | "resume"
N_GPUS = torch.cuda.device_count()
MICRO_BATCH = 16                # lower to 12 or 8 if bench shows OOM / > 15 GB peak
GRAD_ACCUM = 8
TIME_LIMIT_H = 10.0
# From bench: MAX_ITERS = (tok_per_sec * TIME_LIMIT_H * 3600) // (MICRO_BATCH * 512 * N_GPUS * GRAD_ACCUM)
MAX_ITERS = 3800
RESUME_PATH = ""                # e.g. "/kaggle/input/02-train/ckpt_latest.pt"
COMPILE = False
# =============================================
# data can arrive as the dataset "pygpt-python-tokens" or as Notebook 01's mounted output
import glob
DATA_DIR = "/kaggle/input/pygpt-python-tokens"
if MODE != "sanity" and not os.path.exists(DATA_DIR + "/train.bin"):
    hits = glob.glob("/kaggle/input/**/train.bin", recursive=True)
    assert hits, "add the pygpt-python-tokens dataset or Notebook 01's output as an input"
    DATA_DIR = os.path.dirname(hits[0])
print("DATA_DIR =", DATA_DIR)
TOK_PER_STEP = MICRO_BATCH * 512 * N_GPUS * GRAD_ACCUM
print(f"tokens/step = {TOK_PER_STEP:,}; {MAX_ITERS} steps = {MAX_ITERS*TOK_PER_STEP/1e6:.0f}M tokens")
os.chdir("/kaggle/working/pygpt")

In [ ]:
if MODE == "sanity":
    !python tests/sanity.py
    !python tests/make_synthetic_data.py --out_dir data/synthetic
    !torchrun --standalone --nproc_per_node={N_GPUS} train.py --config config/smoke.py \
        --data_dir=data/synthetic --ckpt_dir=/kaggle/working/smoke --max_iters=60
    print("\nsmoke run finished; loss should have dropped well below 10.4")

In [ ]:
if MODE == "bench":
    # 60 steps of the real config on real data: read tok/s and peak memory from the output
    !torchrun --standalone --nproc_per_node={N_GPUS} train.py --config config/gpt2_small_py.py \
        --data_dir={DATA_DIR} --ckpt_dir=/kaggle/working/bench --micro_batch_size={MICRO_BATCH} \
        --grad_accum_steps={GRAD_ACCUM} --max_iters=60 --eval_interval=30 --eval_iters=4 --warmup_iters=20 \
        --compile={COMPILE}
    # resume test: continue the same run for 20 more steps from ckpt_latest.pt
    !torchrun --standalone --nproc_per_node={N_GPUS} train.py --config config/gpt2_small_py.py \
        --data_dir={DATA_DIR} --ckpt_dir=/kaggle/working/bench --micro_batch_size={MICRO_BATCH} \
        --grad_accum_steps={GRAD_ACCUM} --max_iters=80 --eval_interval=40 --eval_iters=4 --warmup_iters=20 \
        --resume=/kaggle/working/bench/ckpt_latest.pt --compile={COMPILE}
    print("\nIf both finished: note tok/s from the step lines and peak GPU memory from the [eval] lines.")
    print(f"MAX_ITERS = tok_per_sec * {TIME_LIMIT_H*3600:.0f} // {TOK_PER_STEP}")

In [ ]:
if MODE == "main":
    !torchrun --standalone --nproc_per_node={N_GPUS} train.py --config config/gpt2_small_py.py \
        --data_dir={DATA_DIR} --ckpt_dir=/kaggle/working --micro_batch_size={MICRO_BATCH} \
        --grad_accum_steps={GRAD_ACCUM} --max_iters={MAX_ITERS} --time_limit_hours={TIME_LIMIT_H} \
        --compile={COMPILE}

In [ ]:
if MODE == "resume":
    assert RESUME_PATH, "set RESUME_PATH to the previous version's ckpt_latest.pt"
    !cp {RESUME_PATH} /kaggle/working/ckpt_latest.pt
    !torchrun --standalone --nproc_per_node={N_GPUS} train.py --config config/gpt2_small_py.py \
        --data_dir={DATA_DIR} --ckpt_dir=/kaggle/working --micro_batch_size={MICRO_BATCH} \
        --grad_accum_steps={GRAD_ACCUM} --max_iters={MAX_ITERS} --time_limit_hours={TIME_LIMIT_H} \
        --resume=/kaggle/working/ckpt_latest.pt --compile={COMPILE}

In [ ]:
# Loss curve from log.csv (works after main / resume)
import pandas as pd, matplotlib.pyplot as plt, os
log = "/kaggle/working/log.csv"
if os.path.exists(log):
    df = pd.read_csv(log)
    tr = df[df.train_loss.notna()]; va = df[df.val_loss.notna()]
    plt.figure(figsize=(8,4))
    plt.plot(tr.tokens/1e6, tr.train_loss, label="train", alpha=.6)
    plt.plot(va.tokens/1e6, va.val_loss, "o-", label="val")
    plt.xlabel("tokens (M)"); plt.ylabel("loss (nats/token)"); plt.legend(); plt.grid(alpha=.3)
    plt.title("PyGPT training"); plt.savefig("/kaggle/working/loss_curve.png", dpi=150, bbox_inches="tight")
    print(va.tail(5))

In [ ]:
# Quick qualitative check after training
if os.path.exists("/kaggle/working/ckpt_best.pt"):
    !python generate.py --ckpt /kaggle/working/ckpt_best.pt --tokenizer {DATA_DIR}/tokenizer \
        --prompt "def fibonacci(n):\n    \"\"\"Return the nth Fibonacci number.\"\"\"\n" --n 2 --temperature 0.5

### Keeping the checkpoint
Outputs of a committed version (`ckpt_best.pt`, `ckpt_latest.pt`, `log.csv`, `loss_curve.png`) are kept by Kaggle.
To resume in a new version: *Add Input → Your Work → this notebook → previous version*, set `MODE="resume"` and
`RESUME_PATH="/kaggle/input/<slug>/ckpt_latest.pt"`.